In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Labs"
)

xpt_file = DATA_DIR / "TRIGLY_J.xpt"
csv_file = DATA_DIR / "TRIGLY_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 TRIGLY_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTSAF2YR",
    "LBXTR",
    "LBDTRSI",
    "LBDLDL",
    "LBDLDLSI",
    "LBDLDLM",
    "LBDLDMSI",
    "LBDLDLN",
    "LBDLDNSI"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "TRIGLY_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 3036:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 10:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 3,036 rows x 10 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# WHOLE-NUMBER FIELDS
# ------------------------------------------------------------
#
# SEQN
#     respondent identifier
#
# LBXTR
#     triglycerides, mg/dL
#
# LBDLDL
#     Friedewald LDL, mg/dL
#
# LBDLDLM
#     Martin-Hopkins LDL, mg/dL
#
# LBDLDLN
#     NIH Equation 2 LDL, mg/dL
#
#
# GENUINE DECIMAL FIELDS
# ------------------------------------------------------------
#
# WTSAF2YR
#     fasting subsample survey weight
#
# LBDTRSI
#     triglycerides, mmol/L
#
# LBDLDLSI
#     Friedewald LDL, mmol/L
#
# LBDLDMSI
#     Martin-Hopkins LDL, mmol/L
#
# LBDLDNSI
#     NIH Equation 2 LDL, mmol/L
#
#
# IMPORTANT:
#
# Genuine decimal fields are NEVER rounded and NEVER
# converted to Int64.
#

INTEGER_COLUMNS = [
    "SEQN",
    "LBXTR",
    "LBDLDL",
    "LBDLDLM",
    "LBDLDLN"
]


DECIMAL_COLUMNS = [
    "WTSAF2YR",
    "LBDTRSI",
    "LBDLDLSI",
    "LBDLDMSI",
    "LBDLDNSI"
]


if set(
    INTEGER_COLUMNS
    +
    DECIMAL_COLUMNS
) != set(EXPECTED_COLUMNS):

    raise ValueError(
        "Field type plan does not cover "
        "all TRIGLY_J variables."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 5. VERIFY ALL FIELDS LOADED NUMERICALLY
# ============================================================

numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 10 TRIGLY_J variables "
        "to load numerically.\n\n"
        f"Actual numeric columns:\n{numeric_cols}"
    )


print(
    "PASS: All 10 fields loaded numerically."
)


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace this value without validating
# where it occurs.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


tiny_counts = (
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[EXPECTED_COLUMNS]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before repair:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY-VALUE PATTERN
# ============================================================

#
# Attached TRIGLY_J:
#
#     WTSAF2YR = 325 tiny values
#
# These represent fasting survey weights that should
# legitimately be zero.
#

EXPECTED_TINY_COUNTS = {
    "WTSAF2YR": 325
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated TRIGLY_J attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 325:

    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "PASS: Exact WTSAF2YR tiny-zero "
    "pattern validated."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED WTSAF2YR ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

weight_tiny_mask = (
    df["WTSAF2YR"]
    .eq(TINY_VALUE)
)


correction_count = int(
    weight_tiny_mask.sum()
)


if correction_count != 325:

    raise ValueError(
        "Unexpected WTSAF2YR tiny-value count."
    )


df.loc[
    weight_tiny_mask,
    "WTSAF2YR"
] = 0


print("\n--- VALIDATED ZERO RESTORATION ---")

print(
    "WTSAF2YR restored to zero:",
    correction_count
)


if int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
) != 325:

    raise ValueError(
        "WTSAF2YR zero restoration failed."
    )


print(
    "PASS: 325 validated WTSAF2YR zeros restored."
)


# ============================================================
# 9. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

#
# This is the exact expected final numerical source.
#
# ONLY:
#
#     WTSAF2YR tiny -> 0
#
# may differ from source_df.
#

expected_df = source_df.copy()


expected_df.loc[
    expected_df["WTSAF2YR"]
    .eq(TINY_VALUE),
    "WTSAF2YR"
] = 0


# ============================================================
# 10. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[EXPECTED_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny XPORT artifacts remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. DISCOVER FRACTIONAL VALUES
# ============================================================

#
# Run BEFORE any Int64 conversion.
#
# np.round() is used ONLY for comparison.
#
# It DOES NOT modify df.
#

print("\n--- FRACTIONAL VALUE DISCOVERY ---")


fractional_counts = {}


for col in EXPECTED_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        fractional_count = 0

    else:

        fractional_mask = ~np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        fractional_count = int(
            fractional_mask.sum()
        )

    fractional_counts[col] = (
        fractional_count
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


EXPECTED_FRACTIONAL_COUNTS = {
    "SEQN": 0,
    "WTSAF2YR": 2711,
    "LBXTR": 0,
    "LBDTRSI": 2834,
    "LBDLDL": 0,
    "LBDLDLSI": 2775,
    "LBDLDLM": 0,
    "LBDLDMSI": 2774,
    "LBDLDLN": 0,
    "LBDLDNSI": 2805
}


if fractional_counts != EXPECTED_FRACTIONAL_COUNTS:

    raise ValueError(
        "Unexpected fractional-value pattern.\n\n"
        f"Expected:\n{EXPECTED_FRACTIONAL_COUNTS}\n\n"
        f"Actual:\n{fractional_counts}"
    )


print(
    "PASS: Whole-number and decimal "
    "variables identified correctly."
)


# ============================================================
# 12. PROVE INTEGER FIELDS ARE SAFE
# ============================================================

print("\n--- INTEGER FIELD VALIDATION ---")


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        (~whole_mask)
        .sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional values"
    )

    if fractional_count != 0:

        raise ValueError(
            f"{col} unexpectedly contains "
            "fractional values."
        )


print(
    "PASS: All proposed Int64 fields "
    "are truly whole-number."
)


# ============================================================
# 13. VALIDATE SEQN
# ============================================================

print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93708:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93708-102956 validated."
)


# ============================================================
# 14. VALIDATE WTSAF2YR
# ============================================================

positive_weight_mask = (
    df["WTSAF2YR"]
    .gt(0)
)


positive_weight_count = int(
    positive_weight_mask.sum()
)


weight_zero_count = int(
    df["WTSAF2YR"]
    .eq(0)
    .sum()
)


weight_missing_count = int(
    df["WTSAF2YR"]
    .isna()
    .sum()
)


weight_min_positive = float(
    df.loc[
        positive_weight_mask,
        "WTSAF2YR"
    ]
    .min()
)


weight_max_positive = float(
    df.loc[
        positive_weight_mask,
        "WTSAF2YR"
    ]
    .max()
)


print("\n--- WTSAF2YR CHECK ---")

print(
    "Positive weights:",
    positive_weight_count
)

print(
    "Zero weights:",
    weight_zero_count
)

print(
    "Missing weights:",
    weight_missing_count
)

print(
    "Minimum positive weight:",
    repr(
        weight_min_positive
    )
)

print(
    "Maximum positive weight:",
    repr(
        weight_max_positive
    )
)


if positive_weight_count != 2711:

    raise ValueError(
        "Unexpected WTSAF2YR positive count."
    )


if weight_zero_count != 325:

    raise ValueError(
        "Unexpected WTSAF2YR zero count."
    )


if weight_missing_count != 0:

    raise ValueError(
        "Unexpected missing WTSAF2YR values."
    )


if not np.isclose(
    weight_min_positive,
    9133.518063,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected minimum positive WTSAF2YR."
    )


if not np.isclose(
    weight_max_positive,
    944153.249752,
    rtol=0,
    atol=0
):

    raise ValueError(
        "Unexpected maximum positive WTSAF2YR."
    )


print(
    "PASS: WTSAF2YR distribution validated."
)


# ============================================================
# 15. LAB RANGE CHECK HELPER
# ============================================================

def check_range(
    column,
    expected_nonmissing,
    expected_missing,
    expected_min,
    expected_max
):

    nonmissing = int(
        df[column]
        .notna()
        .sum()
    )

    missing = int(
        df[column]
        .isna()
        .sum()
    )

    minimum = float(
        df[column]
        .min()
    )

    maximum = float(
        df[column]
        .max()
    )


    print(
        f"\n--- CHECK: {column} ---"
    )

    print(
        "Nonmissing:",
        nonmissing
    )

    print(
        "Missing:",
        missing
    )

    print(
        "Minimum:",
        minimum
    )

    print(
        "Maximum:",
        maximum
    )


    if nonmissing != expected_nonmissing:

        raise ValueError(
            f"{column} nonmissing count differs."
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs."
        )


    if not np.isclose(
        minimum,
        expected_min,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{column} minimum differs."
        )


    if not np.isclose(
        maximum,
        expected_max,
        rtol=0,
        atol=0
    ):

        raise ValueError(
            f"{column} maximum differs."
        )


    print(
        f"PASS: {column} range/count check."
    )


# ============================================================
# 16. LAB RANGE CHECKS
# ============================================================

check_range(
    "LBXTR",
    2834,
    202,
    10,
    2684
)


check_range(
    "LBDTRSI",
    2834,
    202,
    0.113,
    30.302
)


check_range(
    "LBDLDL",
    2808,
    228,
    18,
    357
)


check_range(
    "LBDLDLSI",
    2808,
    228,
    0.465,
    9.232
)


check_range(
    "LBDLDLM",
    2808,
    228,
    21,
    358
)


check_range(
    "LBDLDMSI",
    2808,
    228,
    0.543,
    9.258
)


check_range(
    "LBDLDLN",
    2827,
    209,
    20,
    359
)


check_range(
    "LBDLDNSI",
    2827,
    209,
    0.517,
    9.284
)


# ============================================================
# 17. MG/DL / SI MISSING-POSITION CHECKS
# ============================================================

PAIR_COLUMNS = [
    (
        "LBXTR",
        "LBDTRSI"
    ),
    (
        "LBDLDL",
        "LBDLDLSI"
    ),
    (
        "LBDLDLM",
        "LBDLDMSI"
    ),
    (
        "LBDLDLN",
        "LBDLDNSI"
    )
]


print(
    "\n--- MG/DL / SI MISSING ALIGNMENT ---"
)


for mg_col, si_col in PAIR_COLUMNS:

    mismatch_count = int(
        np.sum(
            df[mg_col]
            .isna()
            .to_numpy()
            !=
            df[si_col]
            .isna()
            .to_numpy()
        )
    )

    print(
        f"{mg_col} vs {si_col}:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise ValueError(
            f"{mg_col}/{si_col} "
            "missing positions differ."
        )


print(
    "PASS: All unit-pair missing positions match."
)


# ============================================================
# 18. TRIGLYCERIDE SI CONVERSION CONSISTENCY
# ============================================================

#
# LBDTRSI = LBXTR * 0.01129
#
# CHECK ONLY.
#
# DO NOT overwrite LBDTRSI.
#

trig_pair = (
    df[
        [
            "LBXTR",
            "LBDTRSI"
        ]
    ]
    .dropna()
)


trig_expected = (
    trig_pair["LBXTR"]
    *
    0.01129
)


trig_difference = (
    trig_pair["LBDTRSI"]
    -
    trig_expected
).abs()


trig_max_difference = float(
    trig_difference.max()
)


print(
    "\n--- TRIGLYCERIDE SI CONVERSION CHECK ---"
)

print(
    "Records checked:",
    len(trig_pair)
)

print(
    "Maximum absolute difference:",
    trig_max_difference
)


if trig_max_difference > (
    0.0005
    +
    1e-12
):

    raise ValueError(
        "LBDTRSI inconsistent with "
        "LBXTR * 0.01129."
    )


print(
    "PASS: LBDTRSI conversion is consistent."
)


# ============================================================
# 19. LDL SI CONVERSION CONSISTENCY
# ============================================================

#
# LDL SI conversion:
#
#     mg/dL * 0.02586
#
# CHECK ONLY.
#

LDL_CONVERSION_PAIRS = [
    (
        "LBDLDL",
        "LBDLDLSI"
    ),
    (
        "LBDLDLM",
        "LBDLDMSI"
    ),
    (
        "LBDLDLN",
        "LBDLDNSI"
    )
]


print(
    "\n--- LDL SI CONVERSION CHECK ---"
)


for mg_col, si_col in LDL_CONVERSION_PAIRS:

    pair_df = (
        df[
            [
                mg_col,
                si_col
            ]
        ]
        .dropna()
    )

    expected_si = (
        pair_df[mg_col]
        *
        0.02586
    )

    difference = (
        pair_df[si_col]
        -
        expected_si
    ).abs()

    max_difference = float(
        difference.max()
    )

    print(
        f"{mg_col} -> {si_col}:",
        max_difference
    )

    if max_difference > (
        0.0005
        +
        1e-12
    ):

        raise ValueError(
            f"{si_col} inconsistent with "
            f"{mg_col} * 0.02586."
        )


print(
    "PASS: LDL SI variables are consistent."
)


# ============================================================
# 20. LDL EQUATION AVAILABILITY CHECK
# ============================================================

#
# Friedewald:
#     LBXTR < 400
#
# Martin-Hopkins:
#     LBXTR < 400
#
# NIH Equation 2:
#     LBXTR < 800
#

equation_rules = [
    (
        "LBDLDL",
        400
    ),
    (
        "LBDLDLM",
        400
    ),
    (
        "LBDLDLN",
        800
    )
]


print(
    "\n--- LDL EQUATION AVAILABILITY CHECK ---"
)


for column, triglyceride_limit in equation_rules:

    expected_present = (
        df["LBXTR"]
        .notna()
        &
        df["LBXTR"]
        .lt(
            triglyceride_limit
        )
    )


    actual_present = (
        df[column]
        .notna()
    )


    mismatch_count = int(
        (
            expected_present
            !=
            actual_present
        )
        .sum()
    )


    print(
        f"{column} "
        f"(LBXTR < {triglyceride_limit}):",
        f"{mismatch_count} mismatches"
    )


    if mismatch_count != 0:

        raise ValueError(
            f"{column} availability does not "
            f"match expected threshold."
        )


print(
    "PASS: LDL equation availability rules match."
)


# ============================================================
# 21. DECIMAL VARIABLE PRE-CONVERSION FIDELITY
# ============================================================

#
# Check all genuine decimal variables before Int64
# conversion.
#
# WTSAF2YR differs from source_df ONLY where the
# validated 325 zero restorations were performed.
#

print(
    "\n--- DECIMAL VARIABLE PRE-CONVERSION FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Decimal fields changed unexpectedly:\n"
        + str(decimal_failures)
    )


print(
    "PASS: Decimal fields unchanged apart from "
    "validated WTSAF2YR zeros."
)


# ============================================================
# 22. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert only:
#
#     SEQN
#     LBXTR
#     LBDLDL
#     LBDLDLM
#     LBDLDLN
#
# DO NOT TOUCH DECIMAL_COLUMNS.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        raise ValueError(
            f"{col} cannot safely be converted "
            "to Int64."
        )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 5 whole-number fields converted "
    "to nullable Int64."
)

print(
    "PASS: 5 genuine decimal fields "
    "left untouched."
)


# ============================================================
# 23. COMPLETE VALUE FIDELITY BEFORE EXPORT
# ============================================================

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 10 variables match exact "
    "expected corrected XPT values."
)


# ============================================================
# 24. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            original_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing-value positions preserved."
)


# ============================================================
# 25. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if count > 0:

        final_zero_counts[col] = count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


EXPECTED_FINAL_ZERO_COUNTS = {
    "WTSAF2YR": 325
}


if (
    final_zero_counts
    !=
    EXPECTED_FINAL_ZERO_COUNTS
):

    raise ValueError(
        "Unexpected final zero pattern.\n\n"
        f"Expected:\n{EXPECTED_FINAL_ZERO_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly correct."
)


# ============================================================
# 26. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     conversion of decimal columns to Int64
#
# NO:
#
#     recalculation of existing laboratory SI values
#
#
# Genuine decimal values must be serialized naturally
# from their existing pandas/Python float representation.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 27. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# THIS IS CRITICAL.
#
# Exact floating-point fidelity should be checked using:
#
#     float_precision="round_trip"
#
# NOT by reading raw text and then calling:
#
#     pd.to_numeric(...)
#
# because pd.to_numeric() can reconstruct a slightly
# different IEEE-754 float from the exact same decimal
# string.
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 28. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric "
        "fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 10 CSV variables round-trip "
    "to exact expected corrected XPT values."
)


# ============================================================
# 29. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\n--- CSV MISSING-VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing positions changed during export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 30. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# RAW TEXT IS USED ONLY FOR:
#
#     .0 formatting checks
#     literal zero checks
#     scientific-notation checks
#
# It is NOT used for exact binary-float fidelity.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    3036,
    10
):

    raise ValueError(
        "Raw CSV dimensions changed."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "Raw CSV column order changed."
    )


# ============================================================
# 31. INTEGER-FIELD .0 CHECK
# ============================================================

#
# ONLY check proven whole-number fields.
#
# Do NOT check genuine decimal fields.
#

print("\n--- INTEGER-FIELD .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "whole-number fields."
)


# ============================================================
# 32. DECIMAL FIELD RAW-TEXT CHECK
# ============================================================

#
# Do NOT enforce cosmetic decimal formatting.
#
# Examples:
#
#     4.0
#     4
#     4.000
#
# can all represent the same numeric value depending
# on serialization.
#
# Exact numeric fidelity is determined from roundtrip_df.
#

print("\n--- DECIMAL FIELD CSV CHECK ---")


for col in DECIMAL_COLUMNS:

    nonblank_count = int(
        raw_csv_df[col]
        .ne("")
        .sum()
    )

    print(
        f"{col} populated:",
        nonblank_count
    )


print(
    "PASS: Decimal fields retained without "
    "forced formatting."
)


# ============================================================
# 33. WTSAF2YR PRECISION DIAGNOSTIC
# ============================================================

#
# This specifically protects against the false failure
# previously encountered.
#
# Example from this dataset:
#
# Original XPT float:
#
#     31942.302908999998
#
# CSV text:
#
#     31942.302908999998
#
# pd.to_numeric() may parse that as:
#
#     31942.302909
#
# producing a tiny ~3.64e-12 binary difference.
#
# float_precision="round_trip" correctly reconstructs
# the original float.
#

expected_weight_values = (
    expected_df["WTSAF2YR"]
    .to_numpy(
        dtype=float
    )
)


roundtrip_weight_values = (
    roundtrip_df["WTSAF2YR"]
    .to_numpy(
        dtype=float
    )
)


weight_roundtrip_exact = np.allclose(
    expected_weight_values,
    roundtrip_weight_values,
    rtol=0,
    atol=0,
    equal_nan=True
)


print(
    "\n--- WTSAF2YR ROUND-TRIP PRECISION ---"
)

print(
    "Exact XPT -> CSV -> float round-trip:",
    weight_roundtrip_exact
)


if not weight_roundtrip_exact:

    raise ValueError(
        "WTSAF2YR genuinely changed "
        "during CSV export."
    )


print(
    "PASS: WTSAF2YR survey weights "
    "preserved exactly."
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

#
# Here pd.to_numeric() is safe because we are only
# checking whether a raw CSV token numerically equals
# the known tiny artifact.
#
# We are NOT performing exact source-float equality.
#

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY-VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 35. FINAL CSV ZERO PATTERN
# ============================================================

#
# pd.to_numeric() is also appropriate here because
# we are simply classifying literal/numeric zero.
#

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values.eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if (
    csv_zero_counts
    !=
    EXPECTED_FINAL_ZERO_COUNTS
):

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_FINAL_ZERO_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final CSV contains exactly "
    "325 WTSAF2YR zeros."
)


# ============================================================
# 36. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    count = int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


    if count > 0:

        scientific_errors[col] = count


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    scientific_errors
)


if scientific_errors:

    raise ValueError(
        "Unexpected scientific notation found:\n"
        + str(scientific_errors)
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 37. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93708:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN 93708-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 38. FINAL DECIMAL-FIELD EXACT FIDELITY
# ============================================================

#
# IMPORTANT CORRECTION
# ------------------------------------------------------------
#
# Use roundtrip_df here.
#
# DO NOT use:
#
#     pd.to_numeric(raw_csv_df[col])
#
# for exact binary-float equality.
#
# float_precision="round_trip" is specifically designed
# to reconstruct the exported floating-point values
# faithfully.
#

print(
    "\n--- FINAL DECIMAL-FIELD EXACT FIDELITY ---"
)


decimal_csv_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    print(
        f"{col}:",
        "PASS" if same else "FAIL"
    )


    if not same:

        decimal_csv_failures.append(
            col
        )


if decimal_csv_failures:

    raise ValueError(
        "Decimal fidelity failures:\n"
        + str(decimal_csv_failures)
    )


print(
    "PASS: Every genuine decimal field "
    "round-trips exactly."
)


# ============================================================
# 39. FINAL ALL-COLUMN SOURCE -> CSV FIDELITY
# ============================================================

#
# Same correction:
#
# use roundtrip_df rather than reparsing raw strings.
#

print(
    "\n--- FINAL ALL-COLUMN SOURCE -> CSV FIDELITY ---"
)


final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves all expected "
    "TRIGLY_J numeric values exactly."
)


# ============================================================
# 40. OPTIONAL FLOAT DIFFERENCE DIAGNOSTIC
# ============================================================

#
# This is useful if you ever want to see whether ANY
# field differs at binary-float level after round-trip.
#
# Expected result:
#
#     maximum difference = 0
#

print(
    "\n--- ROUND-TRIP MAXIMUM ABSOLUTE DIFFERENCES ---"
)


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    final_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    valid_mask = (
        ~np.isnan(expected_values)
        &
        ~np.isnan(final_values)
    )


    if valid_mask.any():

        max_diff = float(
            np.max(
                np.abs(
                    expected_values[valid_mask]
                    -
                    final_values[valid_mask]
                )
            )
        )

    else:

        max_diff = 0.0


    print(
        f"{col}: {max_diff}"
    )


# ============================================================
# 41. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "TRIGLY_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Laboratory - "
    "LDL Cholesterol & Triglycerides"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "WTSAF2YR zero restorations:",
    correction_count
)

print(
    "Whole-number fields:",
    INTEGER_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Positive fasting weights:",
    positive_weight_count
)

print(
    "Zero fasting weights:",
    weight_zero_count
)

print(
    "WTSAF2YR minimum positive:",
    repr(
        weight_min_positive
    )
)

print(
    "WTSAF2YR maximum positive:",
    repr(
        weight_max_positive
    )
)

print(
    "LBXTR nonmissing:",
    int(
        df["LBXTR"]
        .notna()
        .sum()
    )
)

print(
    "LBXTR range:",
    f"{df['LBXTR'].min()}-"
    f"{df['LBXTR'].max()}"
)

print(
    "LBDTRSI range:",
    f"{df['LBDTRSI'].min()}-"
    f"{df['LBDTRSI'].max()}"
)

print(
    "Friedewald LDL records:",
    int(
        df["LBDLDL"]
        .notna()
        .sum()
    )
)

print(
    "Martin-Hopkins LDL records:",
    int(
        df["LBDLDLM"]
        .notna()
        .sum()
    )
)

print(
    "NIH Equation 2 LDL records:",
    int(
        df["LBDLDLN"]
        .notna()
        .sum()
    )
)

print(
    "Exact decimal round-trip failures: "
    "0 expected"
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0' in integer fields: "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 TRIGLY_J XPT -> CSV
Rows: 3,036
Columns: 10
PASS: Structure = 3,036 rows x 10 columns.

--- FIELD TYPE PLAN ---
Whole-number fields: ['SEQN', 'LBXTR', 'LBDLDL', 'LBDLDLM', 'LBDLDLN']
Genuine decimal fields: ['WTSAF2YR', 'LBDTRSI', 'LBDLDLSI', 'LBDLDMSI', 'LBDLDNSI']
PASS: All 10 fields loaded numerically.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before repair: 0
Tiny XPORT artifacts: 325

Tiny values by variable:
WTSAF2YR    325
PASS: Exact 325-value WTSAF2YR tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
WTSAF2YR restored to zero: 325
PASS: 325 validated survey-weight zeros restored.

Tiny XPORT artifacts remaining: 0
PASS: No tiny XPORT artifacts remain.

--- FRACTIONAL VALUE DISCOVERY ---
SEQN: 0 fractional observations
WTSAF2YR: 2,711 fractional observations
LBXTR: 0 fractional observations
LBDTRSI: 2,834 fractional observations
LBDLDL: 0 fractional observations
LBDLDLSI: 2,775 fractional observations
LBDLDLM: 0 fractional observati

ValueError: Decimal fidelity failures:
['WTSAF2YR']